# EGM Veri Toplama — BES Kalıcılık ve Rekabet Projesi

Emeklilik Gözetim Merkezi (egm.org.tr) istatistik raporlarını tarih tarih indirir.

**Hedef raporlar**

| Kod | Rapor | Ne işe yarayacak |
|---|---|---|
| `GR-01-015` | Sözleşmelerin Yürürlüğe Giriş Ayına Göre Sonlanma Oranı | Kohort retention eğrisi (2003.10 → bugün) |
| `GR-01-010` | Sonlanan Sözleşmelerin Sonlanma Nedenine Göre Dağılımı | Ayrılma nedenleri kırılımı |
| `GR-01-019` | Cayma Hakkı Kullanım Oranı | Cayma trendi |
| `EGM-05-006` | Emeklilik Şirketi ve Faiz Detayında Fon Büyüklükleri | Şirket bazında pazar payı — MetLife vs rakipler |
| `EGM-05-001` | Fon Bazında Fon İşletim Gideri Kesinti Oranları | Maliyet rekabeti |

**Çalışma sırası:** Kurulum → Script'i yaz → Keşif (`--probe`) → Küçük deneme → Tam toplama → İndir

---

> **Kaynak:** Bu defterde toplanan tüm istatistiksel veriler Emeklilik Gözetim Merkezi A.Ş.'ye (EGM) aittir. Kaynak gösterilmeden alıntı yapılamaz. Projenin README'sinde EGM'yi kaynak olarak belirt.

## 1. Kurulum

Colab oturumu her sıfırlandığında bu hücreyi tekrar çalıştırman gerekir.

In [ ]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

## 2. Toplayıcı script'i

Script repo'da duruyor, buraya kopyalamak yerine oradan çekiyoruz — tek doğru
nüsha repo olsun, iki yerde iki farklı kopya dolaşmasın.

Script'i neden ayrı bir dosya olarak çalıştırıyoruz: Playwright'ın sync API'si
notebook'un asyncio döngüsüyle çakışıyor. `!python` ile ayrı süreç olarak
çalıştırınca sorun kalmıyor.

In [ ]:
!wget -q -O egm_collector.py https://raw.githubusercontent.com/esero88/eser-portfolio/main/pension-retention-analytics/scripts/egm_collector.py
!wc -l egm_collector.py

## 3. Keşif — önce ne olduğunu görelim

`--probe` hiçbir şey indirmez. Sadece şunu söyler:
- Tarih listesini bulabildi mi, kaç tarih var
- "Excel'e Aktar" düğmesini görebildi mi

**Yapman gereken:** EGM'de raporu aç, tarayıcının adres çubuğundaki linki kopyala,
aşağıdaki `RAPOR_URL` değişkenine yapıştır.

In [ ]:
RAPOR_URL = ""   # <-- raporun linkini buraya yapıştır
RAPOR_KODU = "GR-01-015"

assert RAPOR_URL, "Önce RAPOR_URL'i doldur"
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --probe

### Çıktıyı nasıl okuyacaksın

- **"N tarih bulundu"** + **"Excel düğmesi: BULUNDU"** → her şey yolunda, 4. adıma geç.
- **"tarih listesi bulunamadı"** → tarih seçici normal bir `<select>` değil.
  Script'teki `tarih_secicisini_bul()` fonksiyonunu güncellememiz lazım, çıktıyı bana at.
- **"Excel düğmesi: BULUNAMADI"** → `export_dugmesini_bul()` güncellenecek, yine çıktıyı at.
- **Zaman aşımı / bağlantı hatası** → EGM, Colab'ın IP'sini engelliyor olabilir.
  Bu durumda script'i kendi bilgisayarında çalıştırman gerekir.

## 4. Küçük deneme

Her şeyi indirmeden önce 3 tarihle dene. Dosyalar gerçekten iniyor mu, içleri dolu mu?

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out data/raw --limit 3

In [ ]:
import pandas as pd
from pathlib import Path

dosyalar = sorted(Path(f"data/raw/{RAPOR_KODU}").glob("*.xlsx"))
print(f"{len(dosyalar)} dosya indi:")
for d in dosyalar:
    print(" ", d.name, f"({d.stat().st_size:,} byte)")

if dosyalar:
    ornek = pd.read_excel(dosyalar[0])
    print(f"\n{dosyalar[0].name}  ->  {ornek.shape[0]} satır × {ornek.shape[1]} kolon")
    display(ornek.head(10))

## 5. Tam toplama

Deneme sorunsuzsa hepsini indir. Zaten inmiş dosyaları atlar,
yarıda kesilirse tekrar çalıştırınca kaldığı yerden devam eder.

`--delay 2.5` saniyelik bekleme siteyi yormamak için. Düşürme.

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out data/raw --delay 2.5

## 6. Diğer raporlar

Her raporun kendi linki var. Sırayla aynı işlemi yap:
linki kopyala, kodu değiştir, çalıştır.

In [ ]:
# Örnek: şirket bazında fon büyüklükleri
RAPOR_URL_2 = ""   # <-- EGM-05-006'nın linki
RAPOR_KODU_2 = "EGM-05-006"

if RAPOR_URL_2:
    !python egm_collector.py --url "$RAPOR_URL_2" --code "$RAPOR_KODU_2" --out data/raw --delay 2.5

## 7. Toplananları indir

Colab oturumu kapanınca dosyalar silinir. İki seçenek: bilgisayarına indir ya da Drive'a kaydet.

In [ ]:
# Seçenek A — zip'leyip bilgisayarına indir
from google.colab import files
!zip -rq egm_data.zip data/raw
files.download("egm_data.zip")

In [ ]:
# Seçenek B — Google Drive'a kaydet
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/egm-bes-projesi"
# !cp -r data/raw "/content/drive/MyDrive/egm-bes-projesi/"

---

## Sonraki adım

Ham dosyalar elimize geçtikten sonra:

1. **Temizlik** — EGM-05-006'nın başlıkları iki satıra bölünmüş geliyor, sayılar bilimsel gösterimde.
   Bunları düzleştirip tek bir uzun formata (`tarih, şirket, segment, tutar`) çevireceğiz.
2. **Ambar** — temiz tabloları SQL'e yükleyip yıldız şema kuracağız.
3. **Dashboard** — Power BI: kohort retention eğrisi, pazar payı, MetLife'ın konumu.
4. **Agent** — metrik sözlüğü + doğal dilden SQL, 30 soruluk doğruluk testi.